# Day-5 Practice Tasks

Practice exercises spanning the Day-5 topics: regular expressions and concurrency (threading &
multiprocessing).

- Task 1 (Easy): Phone number validator
- Task 2 (Easy): IPv4 address validator
- Task 3 (Easy): Password strength validator
- Task 4 (Medium): Single-thread vs two-thread timing
- Task 5 (Medium): Multiprocessing Pool speedup
- Task 6 (Medium): Thread synchronization with Lock

## Task 1 (Easy): Phone Number Validator

Write a function `is_valid_phone(number)` that validates Indian mobile numbers: 10 digits
starting with 6-9, with an optional `+91`, `91`, or `0` prefix (optionally followed by a space
or hyphen).

Test it against a mix of valid and invalid numbers.

In [1]:
# Solution
import re

def is_valid_phone(number):
    pattern = r'^(?:\+91[-\s]?|91[-\s]?|0)?[6-9]\d{9}$'
    return bool(re.match(pattern, number))

test_numbers = [
    "9876543210",        # valid
    "+91-9876543210",    # valid
    "919876543210",      # valid
    "09876543210",       # valid
    "1876543210",        # invalid - starts with 1
    "98765432",          # invalid - too short
    "98765432101",       # invalid - too long
    "+91-1234567890",    # invalid - starts with 1 after prefix
]

for num in test_numbers:
    print(f"{num!r:20} -> {is_valid_phone(num)}")

'9876543210'         -> True
'+91-9876543210'     -> True
'919876543210'       -> True
'09876543210'        -> True
'1876543210'         -> False
'98765432'           -> False
'98765432101'        -> False
'+91-1234567890'     -> False


## Task 2 (Easy): IPv4 Address Validator

Write a function `is_valid_ipv4(ip)` that validates an IPv4 address: four dot-separated octets,
each between 0 and 255.

Test it against a mix of valid and invalid addresses.

In [2]:
# Solution
import re

def is_valid_ipv4(ip):
    octet = r'(25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)'
    pattern = rf'^{octet}\.{octet}\.{octet}\.{octet}$'
    return bool(re.match(pattern, ip))

test_ips = [
    "192.168.1.1",       # valid
    "0.0.0.0",           # valid
    "255.255.255.255",   # valid
    "256.1.1.1",         # invalid - 256 out of range
    "192.168.1",         # invalid - only 3 octets
    "192.168.1.1.1",     # invalid - 5 octets
    "192.168.1.-1",      # invalid - negative
    "abc.def.gha.bcd",   # invalid - not numeric
]

for ip in test_ips:
    print(f"{ip!r:20} -> {is_valid_ipv4(ip)}")

'192.168.1.1'        -> True
'0.0.0.0'            -> True
'255.255.255.255'    -> True
'256.1.1.1'          -> False
'192.168.1'          -> False
'192.168.1.1.1'      -> False
'192.168.1.-1'       -> False
'abc.def.gha.bcd'    -> False


## Task 3 (Easy): Password Strength Validator

Write a function `is_strong_password(password)` that requires:
- at least 8 characters
- at least one lowercase letter
- at least one uppercase letter
- at least one digit
- at least one special character from `!@#$%^&*`

Test it against passwords that each fail one rule, plus one that passes all of them.

In [3]:
# Solution
import re

def is_strong_password(password):
    pattern = r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[!@#$%^&*]).{8,}$'
    return bool(re.match(pattern, password))

test_passwords = [
    "Weakpass1!",       # valid
    "weakpass1!",       # invalid - no uppercase
    "WEAKPASS1!",       # invalid - no lowercase
    "Weakpassword!",    # invalid - no digit
    "Weakpass1",        # invalid - no special char
    "W1!a",             # invalid - too short
]

for pwd in test_passwords:
    print(f"{pwd!r:20} -> {is_strong_password(pwd)}")

'Weakpass1!'         -> True
'weakpass1!'         -> False
'WEAKPASS1!'         -> False
'Weakpassword!'      -> False
'Weakpass1'          -> False
'W1!a'               -> False


## Task 4 (Medium): Single-Thread vs Two-Thread Timing

Simulate 4 units of I/O-bound work (e.g. network/file waits) with `time.sleep(1)` each.

1. Run all 4 units sequentially in the main thread and time it.
2. Split the same 4 units across 2 `threading.Thread`s (2 units each), run them concurrently,
   `join()` both, and time it.
3. Print both durations and the speedup - the two-thread version should take roughly half the
   time, since threads release the GIL while sleeping/waiting on I/O.

In [4]:
# Solution
import time
import threading

def worker(n):
    for _ in range(n):
        time.sleep(1)

# 1. Single thread: 4 units of work, one after another
start = time.perf_counter()
worker(4)
single_thread_time = time.perf_counter() - start
print(f"Single-thread time: {single_thread_time:.2f}s")

# 2. Two threads: 2 units of work each, running concurrently
start = time.perf_counter()
t1 = threading.Thread(target=worker, args=(2,))
t2 = threading.Thread(target=worker, args=(2,))
t1.start()
t2.start()
t1.join()
t2.join()
two_thread_time = time.perf_counter() - start
print(f"Two-thread time:    {two_thread_time:.2f}s")

print(f"Speedup: {single_thread_time / two_thread_time:.2f}x")

Single-thread time: 4.00s


Two-thread time:    2.00s
Speedup: 2.00x


## Task 5 (Medium): Multiprocessing Pool Speedup

Using the CPU-bound `count_primes_below(n)` from `mp_worker.py` (kept in a separate module so
Windows can pickle it for child processes), compute prime counts below `1,000,000` for 4
different inputs:

1. Sequentially, in a plain `for` loop, timed.
2. In parallel with `multiprocessing.Pool(processes=2).map(...)`, timed.

Print both durations, the results (to confirm they match), and the speedup.

In [5]:
# Solution
import time
from multiprocessing import Pool
from mp_worker import count_primes_below

inputs = [1_000_000, 1_000_000, 1_000_000, 1_000_000]

# 1. Sequential
start = time.perf_counter()
sequential_results = [count_primes_below(n) for n in inputs]
sequential_time = time.perf_counter() - start
print(f"Sequential time: {sequential_time:.2f}s -> {sequential_results}")

# 2. Parallel with a Pool of 2 processes
if __name__ == "__main__":
    start = time.perf_counter()
    with Pool(processes=2) as pool:
        pool_results = pool.map(count_primes_below, inputs)
    pool_time = time.perf_counter() - start
    print(f"Pool time:        {pool_time:.2f}s -> {pool_results}")
    print(f"Speedup: {sequential_time / pool_time:.2f}x")

Sequential time: 33.31s -> [78498, 78498, 78498, 78498]


Pool time:        10.05s -> [78498, 78498, 78498, 78498]
Speedup: 3.31x


## Task 6 (Medium): Thread Synchronization with Lock

1. Create a shared `counter = 0` and have 4 threads each increment it 2,000 times, *without*
   any synchronization. Each increment reads the counter, then calls `time.sleep(0)` before
   writing the new value back - this forces the GIL to switch to another thread between the
   read and the write, so interleaving (and lost updates) is guaranteed rather than left to
   chance. Print the final value and show it's far less than the expected `8,000`.
2. Repeat the same experiment, but guard the read-sleep-write with a `threading.Lock()`. Print
   the final value and show it now matches `8,000` exactly.

In [6]:
# Solution
import threading
import time

NUM_THREADS = 4
INCREMENTS = 2_000

# 1. Without a lock - race condition. time.sleep(0) forces a context switch
# between the read and the write, so the interleaving (and lost updates) is
# guaranteed instead of being a rare timing accident.
counter = 0

def increment_unsafe():
    global counter
    for _ in range(INCREMENTS):
        temp = counter
        time.sleep(0)
        counter = temp + 1

threads = [threading.Thread(target=increment_unsafe) for _ in range(NUM_THREADS)]
for t in threads:
    t.start()
for t in threads:
    t.join()

expected = NUM_THREADS * INCREMENTS
print(f"Without lock: counter = {counter}, expected = {expected}, matches = {counter == expected}")

# 2. With a lock - the read-sleep-write is now atomic, so no updates are lost
counter = 0
lock = threading.Lock()

def increment_safe():
    global counter
    for _ in range(INCREMENTS):
        with lock:
            temp = counter
            time.sleep(0)
            counter = temp + 1

threads = [threading.Thread(target=increment_safe) for _ in range(NUM_THREADS)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"With lock:    counter = {counter}, expected = {expected}, matches = {counter == expected}")

Without lock: counter = 2438, expected = 8000, matches = False
With lock:    counter = 8000, expected = 8000, matches = True
